## Demo - Step by Step 

**Step 1: Create Target Data**

In [0]:
target_data = [
    (101, "John", 50000),
    (102, "David", 60000),
    (103, "Mike", 70000)
]

target_df = spark.createDataFrame(target_data, ["emp_id", "name", "salary"])

target_df.write.mode("overwrite").saveAsTable("ytdemo.data.employee_delta")

target_df.display()

emp_id,name,salary
101,John,50000
102,David,60000
103,Mike,70000


**Step 2: Create the Source Data**

In [0]:
source_data = [
    (102, "David", 65000),
    (103, "Mike", 75000),
    (104, "Alex", 55000)
]

source_df = spark.createDataFrame(source_data, ["emp_id", "name", "salary"])

source_df.display()

emp_id,name,salary
102,David,65000
103,Mike,75000
104,Alex,55000


### Step 3: Import Delta Table

In [0]:
from delta.tables import DeltaTable

# The DeltaTable class gives access to powerfull operations like merge, update, Delete , History, and more

### Step 4: Load Existing Delta Table

In [0]:
dt = DeltaTable.forName(spark, "ytdemo.data.employee_delta")
type(dt)
dt.toDF().display()

emp_id,name,salary
101,John,50000
102,David,65000
103,Mike,75000
104,Alex,55000


**Step 5: Perform the MERGE Operation**

In [0]:
dt.alias("target") \
    .merge(
        source_df.alias("source"),
        "target.emp_id = source.emp_id"
    ) \
    .whenMatchedUpdate(
        set = {
            "name": "source.name",
            "salary": "source.salary"
        }
    ) \
    .whenNotMatchedInsert(
        values = {
            "emp_id": "source.emp_id",
            "name": "source.name",
            "salary": "source.salary"
        }
    ) \
    .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

**Step 6: Verify the Results**

In [0]:
result_df = spark.read.table("ytdemo.data.employee_delta")
result_df.display()
target_df.display()


emp_id,name,salary
101,John,50000
102,David,65000
103,Mike,75000
104,Alex,55000


emp_id,name,salary
101,John,50000
102,David,60000
103,Mike,70000


![image_1780762892541.png](./image_1780762892541.png "image_1780762892541.png")

**_MERGE Supports Multiple Conditions_**

Example:

.whenMatchedUpdate(

    condition="source.salary > target.salary",

    set={
        "salary":"source.salary"
    }
    
)

Only update when source salary is higher.

# Thanks for Watching
### Please Subscribe, Like , Share, Comment